## Exploración inicial de los datos

### By:
Bryan Escobar Restrepo

### Date:
2026-08-21

### Description:

Exploración general del dataset `Admission_Predict.csv` (probabilidad de admisión a
un posgrado) con el fin de **entender el esquema, verificar los tipos de datos y
corregir los problemas básicos** que impedirían un análisis posterior correcto.

Alcance de este notebook:

1. Descripción general de los datos.
2. Unificar la forma en que se representan los valores nulos.
3. Convertir cada columna a su tipo correcto y uniforme (numérico, categórico,
   booleano, fecha) y corregir los datos donde haga falta.
4. Almacenar el dataset resultante en `data/02_intermediate/` en formato `.parquet`.

**Fuera de alcance:** el análisis estadístico, las visualizaciones y la imputación de
nulos, que corresponden a los notebooks de `3-analysis` y `4-feat_eng`.

**Criterio de diseño:** el archivo se carga **como texto crudo** (`dtype="object"`,
`na_filter=False`) en vez de dejar que `pandas` infiera. Si se deja inferir, pandas
ya decidió por su cuenta qué es nulo y qué tipo tiene cada columna, y esas decisiones
quedan invisibles. Partiendo del texto tal cual está en el archivo, cada conversión
es una decisión explícita y verificable.

## 📚 Import  libraries

In [1]:
# base libraries for data science
from pathlib import Path

import pandas as pd
import pyarrow as pa

pd.set_option("display.max_columns", 50)

print(f"pandas  {pd.__version__}")
print(f"pyarrow {pa.__version__}")

pandas  3.0.5
pyarrow 25.0.1


## ⚙️ Configuración

Las rutas se resuelven a partir de la raíz del repositorio, no con rutas absolutas,
para que el notebook funcione en cualquier máquina y desde cualquier directorio.

In [2]:
def buscar_raiz_proyecto() -> Path:
    """Sube por el árbol de directorios hasta encontrar la raíz del repositorio."""
    marcadores = (".git", "pyproject.toml")
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if any((candidato / marcador).exists() for marcador in marcadores):
            return candidato
    return actual


DATA_DIR = buscar_raiz_proyecto() / "data"
ARCHIVO_RAW = DATA_DIR / "01_raw" / "Admission_Predict.csv"
ARCHIVO_SALIDA = DATA_DIR / "02_intermediate" / "admisiones_type_fixed.parquet"

# los duplicados exactos se eliminan aqui (la justificacion esta mas abajo)
ELIMINAR_DUPLICADOS = True

ARCHIVO_SALIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"Entrada: {ARCHIVO_RAW.relative_to(ARCHIVO_RAW.parents[2])}")
print(f"Salida : {ARCHIVO_SALIDA.relative_to(ARCHIVO_SALIDA.parents[2])}")

Entrada: data/01_raw/Admission_Predict.csv
Salida : data/02_intermediate/admisiones_type_fixed.parquet


## 💾 Load data

Se hacen dos lecturas del mismo archivo, con propósitos distintos:

| Lectura | Para qué sirve |
|---|---|
| `df_inferido` | Ver qué tipos y qué nulos infiere pandas por defecto (solo diagnóstico) |
| `df_crudo` | Trabajar con el texto tal como está en el archivo (auditoría de nulos y tipos) |

In [3]:
# lectura 1: tipos inferidos automaticamente por pandas (solo diagnostico)
df_inferido = pd.read_csv(ARCHIVO_RAW)

# lectura 2: todo como texto crudo, sin conversion automatica de nulos
df_crudo = pd.read_csv(ARCHIVO_RAW, dtype="object", na_filter=False)

print(f"Dimensiones: {df_crudo.shape[0]} filas x {df_crudo.shape[1]} columnas")

Dimensiones: 623 filas x 8 columnas


## 📊 Descripción general de los datos

In [4]:
df_inferido.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          612 non-null    float64
 1   TOEFL Score        604 non-null    float64
 2   University Rating  617 non-null    float64
 3   SOP                607 non-null    float64
 4   LOR                615 non-null    float64
 5   CGPA               621 non-null    float64
 6   Research           594 non-null    float64
 7   Chance of Admit    623 non-null    float64
dtypes: float64(8)
memory usage: 39.1 KB


In [5]:
df_crudo.sample(10, random_state=42)

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
249,321,111,3,3.5,4,8.83,1,0.77
558,326,112,3,3.5,3,9.05,,0.74
174,321,111,4,4,4,8.97,1,0.87
280,311,102,3,4.5,4,8.64,1,0.68
110,305,108,5,3,3,8.48,0,0.61
244,314,107,2,2.5,4,8.56,0,0.63
228,318,112,3,4,3.5,8.67,0,0.71
227,312,110,2,3.5,3,8.53,0,0.64
465,316,100,2,1.5,3,8.16,1,0.71
148,339,116,4,4,3.5,9.8,1,0.96


Primer problema, y es de los invisibles: **dos nombres de columna traen un espacio al
final** (`'LOR '` y `'Chance of Admit '`). Al imprimir el DataFrame no se nota, pero
`df["LOR"]` lanzaría `KeyError`. Por eso se imprime el `repr()` de cada nombre.

In [6]:
print([repr(columna) for columna in df_crudo.columns])

["'GRE Score'", "'TOEFL Score'", "'University Rating'", "'SOP'", "'LOR '", "'CGPA'", "'Research'", "'Chance of Admit '"]


In [7]:
def resumen_columnas(df: pd.DataFrame, n_ejemplos: int = 4) -> pd.DataFrame:
    """Inventario por columna: tipo inferido, cardinalidad, vacíos y ejemplos."""
    total = len(df)
    filas = []
    for columna in df.columns:
        texto = df[columna].astype("string").str.strip()
        vacios = int((texto == "").sum())
        ejemplos = texto[texto != ""].unique()[:n_ejemplos]
        filas.append(
            {
                "columna": columna,
                "dtype_inferido": str(df_inferido[columna].dtype),
                "n_unicos": int(texto.nunique()),
                "n_vacios": vacios,
                "%_vacios": round(100 * vacios / total, 2),
                "ejemplos": " | ".join(ejemplos),
            }
        )
    return pd.DataFrame(filas)


resumen_columnas(df_crudo)

,columna,dtype_inferido,n_unicos,n_vacios,%_vacios,ejemplos
0,GRE Score,float64,50,11,1.77,337 | 324 | 316 | 322
1,TOEFL Score,float64,30,19,3.05,118 | 107 | 104 | 110
2,University Rating,float64,6,6,0.96,4 | 3 | 2 | 5
3,SOP,float64,10,16,2.57,4.5 | 4 | 3 | 3.5
4,LOR,float64,10,8,1.28,4.5 | 3.5 | 2.5 | 3
5,CGPA,float64,169,2,0.32,9.65 | 8.87 | 8 | 8.67
6,Research,float64,3,29,4.65,1 | 0
7,Chance of Admit,float64,60,0,0.00,0.92 | 0.76 | 0.72 | 0.8


In [8]:
n_duplicados = int(df_crudo.duplicated().sum())
constantes = [columna for columna in df_crudo.columns if df_crudo[columna].nunique() <= 1]

print(f"Filas duplicadas exactas: {n_duplicados} de {len(df_crudo)}")
print(f"Filas unicas            : {len(df_crudo.drop_duplicates())}")
print(f"Columnas constantes     : {constantes}")
print(f"Memoria (lectura cruda) : {df_crudo.memory_usage(deep=True).sum() / 1024:.1f} KB")
print("No hay columna identificadora en el dataset.")

Filas duplicadas exactas: 152 de 623
Filas unicas            : 471
Columnas constantes     : []
Memoria (lectura cruda) : 250.2 KB
No hay columna identificadora en el dataset.


### 📝 Diccionario de datos

El problema: estimar la probabilidad de admisión a un posgrado a partir del perfil del
aspirante. **Una fila = un aspirante**, sin columna identificadora.

| Columna | Descripción | Tipo objetivo | Dominio válido |
|---|---|---|---|
| `GRE Score` | Puntaje del examen GRE | `Int64` | 0 – 340, entero |
| `TOEFL Score` | Puntaje del examen TOEFL | `Int64` | 0 – 120, entero |
| `University Rating` | Calificación de la universidad de origen | `category` ordinal | 1 < 2 < 3 < 4 < 5 |
| `SOP` | Fuerza del *statement of purpose* | `Float64` | 1.0 – 5.0, pasos de 0.5 |
| `LOR ` | Fuerza de las cartas de recomendación | `Float64` | 1.0 – 5.0, pasos de 0.5 |
| `CGPA` | Promedio acumulado del pregrado | `Float64` | 0 – 10 |
| `Research` | ¿Tiene experiencia en investigación? | `boolean` | 0 = No, 1 = Sí |
| `Chance of Admit ` | **Variable objetivo:** probabilidad de admisión | `Float64` | 0.0 – 1.0 |

Los dominios salen de `data/01_raw/Informacion.txt`, no de los datos: sirven como
control externo para detectar valores imposibles.

**Sobre fechas:** el dataset no tiene ninguna columna temporal. Cada fila es una
fotografía del perfil de un aspirante sin marca de tiempo, así que no hay nada que
convertir a `datetime` ni orden cronológico que respetar al partir en train/test más
adelante.

## 🏷️ Normalizar los nombres de columna

Se pasan a `snake_case`, lo que de paso elimina los espacios finales invisibles. Se
valida el resultado contra la lista esperada para que el notebook falle de inmediato
si el archivo de origen cambia de esquema.

In [9]:
def a_snake_case(nombre: str) -> str:
    """Normaliza un nombre de columna: sin espacios sobrantes y en minúsculas."""
    return nombre.strip().lower().replace(" ", "_")


df = df_crudo.rename(columns=a_snake_case)

COLUMNAS_ESPERADAS = [
    "gre_score",
    "toefl_score",
    "university_rating",
    "sop",
    "lor",
    "cgpa",
    "research",
    "chance_of_admit",
]
if list(df.columns) != COLUMNAS_ESPERADAS:
    raise ValueError(f"Los nombres normalizados no coinciden con lo esperado: {list(df.columns)}")

print(list(df.columns))

['gre_score', 'toefl_score', 'university_rating', 'sop', 'lor', 'cgpa', 'research', 'chance_of_admit']


## 🕳️ Unificar la forma como se representan los valores nulos

Antes de asumir cómo viene escrito un nulo, se cuenta. Se busca sobre el **texto
crudo** cualquier variante habitual (`""`, `?`, `NA`, `null`, `-`, `desconocido`, ...),
que es justo lo que la lectura por defecto de pandas habría escondido.

In [10]:
CENTINELAS_NULOS = [
    "",
    "?",
    "-",
    "--",
    ".",
    "na",
    "n/a",
    "#n/a",
    "nan",
    "null",
    "none",
    "missing",
    "unknown",
    "desconocido",
    "sin dato",
    "ninguno",
]


def auditar_centinelas(df: pd.DataFrame, centinelas: list[str]) -> pd.DataFrame:
    """Cuenta, por columna, cada representación de nulo encontrada en el texto."""
    patron = {valor.strip().lower() for valor in centinelas}
    filas = []
    for columna in df.columns:
        conteo = df[columna].astype("string").str.strip().str.lower().value_counts()
        detalle = {valor: int(n) for valor, n in conteo.items() if valor in patron}
        if detalle:
            filas.append(
                {
                    "columna": columna,
                    "representaciones": ", ".join(f"{k!r}: {v}" for k, v in detalle.items()),
                    "total": sum(detalle.values()),
                }
            )
    if not filas:
        return pd.DataFrame(columns=["columna", "representaciones", "total"])
    return pd.DataFrame(filas).sort_values("total", ascending=False)


auditar_centinelas(df, CENTINELAS_NULOS)

,columna,representaciones,total
6,research,'': 29,29
1,toefl_score,'': 19,19
3,sop,'': 16,16
0,gre_score,'': 11,11
4,lor,'': 8,8
2,university_rating,'': 6,6
5,cgpa,'': 2,2


La auditoría confirma que en este dataset **la ausencia de dato se representa de una
sola forma: el campo vacío**. No aparecen `?`, `NA`, `null` ni `-1` como centinelas.

Aun así se aplica la normalización completa sobre toda la lista: deja el criterio
explícito y protege frente a una nueva carga del archivo que sí los traiga.

Los nulos se unifican en **`pd.NA`**, un único centinela válido para cualquier tipo
(entero, flotante, booleano, categórico o texto). Es la diferencia con `np.nan`, que
es un flotante y obliga a promover a `float` cualquier columna entera o booleana que
tenga un solo faltante.

In [11]:
patron_nulos = {valor.strip().lower() for valor in CENTINELAS_NULOS}

for columna in df.columns:
    serie = df[columna].astype("string").str.strip()
    df[columna] = serie.mask(serie.str.lower().isin(patron_nulos), pd.NA)

comparacion_nulos = pd.DataFrame(
    {
        "nulos_lectura_por_defecto": df_inferido.isna().sum().to_numpy(),
        "nulos_tras_unificar": df.isna().sum().to_numpy(),
    },
    index=df.columns,
)
comparacion_nulos["%_nulos"] = (100 * comparacion_nulos["nulos_tras_unificar"] / len(df)).round(2)
comparacion_nulos.sort_values("%_nulos", ascending=False)

,nulos_lectura_por_defecto,nulos_tras_unificar,%_nulos
research,29,29,4.65
toefl_score,19,19,3.05
sop,16,16,2.57
gre_score,11,11,1.77
lor,8,8,1.28
university_rating,6,6,0.96
cgpa,2,2,0.32
chance_of_admit,0,0,0.00


In [12]:
filas_con_nulos = int(df.isna().any(axis=1).sum())
print(f"Celdas nulas          : {int(df.isna().sum().sum())}")
print(
    f"Filas con algun nulo  : {filas_con_nulos} de {len(df)} ({100 * filas_con_nulos / len(df):.1f}%)"
)
print(f"Filas completamente nulas: {int(df.isna().all(axis=1).sum())}")

Celdas nulas          : 91
Filas con algun nulo  : 71 de 623 (11.4%)
Filas completamente nulas: 0


Los nulos están repartidos en 7 de las 8 columnas y afectan al 11 % de las filas. Lo
relevante en esta etapa es que **la variable objetivo `chance_of_admit` no tiene
ningún nulo**: una fila sin objetivo no sirve para entrenar y habría que descartarla.

La **imputación no se hace aquí**. Imputar antes de mirar las distribuciones sesga el
análisis y mete una decisión de modelado en una capa que solo debería tipar los datos;
la estrategia por columna se decide en el notebook de EDA.

## 🔁 Duplicados exactos

152 de las 623 filas (24 %) están repetidas de forma exacta, y ninguna de ellas
contiene nulos, así que no son un artefacto de la unificación anterior.

**Decisión: se eliminan.** Los argumentos:

- Distorsionan cualquier distribución que se calcule después: las filas repetidas
  pesan el doble o el triple sin que haya evidencia de que representen más aspirantes.
- Sin eliminarlas, la misma fila puede caer a la vez en *train* y en *test*, lo que
  infla artificialmente las métricas del modelo (fuga de datos por duplicación).
- El archivo de `01_raw` es inmutable, así que la decisión es reversible: basta con
  poner `ELIMINAR_DUPLICADOS = False` y volver a ejecutar.

**Contraargumento honesto:** sin columna identificadora no se puede *demostrar* que
sean errores de carga y no dos aspirantes con perfiles idénticos. Con 8 variables, dos
de ellas continuas (`cgpa`, `chance_of_admit`), la coincidencia exacta por azar es muy
improbable, y un 24 % de repetición es una tasa típica de una duplicación al construir
el archivo. Queda documentado por si el análisis posterior sugiere lo contrario.

In [13]:
duplicados_con_nulos = int(df[df.duplicated(keep=False)].isna().any(axis=1).sum())
print(f"Filas duplicadas que contienen algun nulo: {duplicados_con_nulos}")

filas_antes = len(df)
if ELIMINAR_DUPLICADOS:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Filas antes : {filas_antes}")
print(f"Filas ahora : {len(df)} (eliminadas {filas_antes - len(df)})")

Filas duplicadas que contienen algun nulo: 0
Filas antes : 623
Filas ahora : 471 (eliminadas 152)


## 🔧 Convertir los datos a su tipo correcto

Todas las columnas están ahora como texto (`string`) con `pd.NA` en los faltantes. La
conversión usa `errors="coerce"`, que convierte en nulo cualquier valor que no encaje;
por eso se guarda el conteo de nulos *antes* y se compara al terminar, para detectar
pérdidas silenciosas.

Se usan los tipos **nullable** de pandas (`Int64`, `Float64`, `boolean`) en vez de los
de numpy, porque son los únicos que admiten `pd.NA` sin cambiar de tipo.

In [14]:
# punto de partida comun: se compara contra "nulos_antes" al terminar la
# conversion, para detectar perdidas silenciosas por errors="coerce"
nulos_antes = df.isna().sum()
df_tipado = df.copy()

### 🏷️ Variables categóricas ordinales

`university_rating` es una escala de 1 a 5 sobre la universidad de origen, no una
cantidad medible: la distancia entre 1 y 2 no tiene por qué ser la misma que entre 4 y
5, y su promedio no significa nada físico. Se declara `category` **ordinal**, con el
orden explícito 1 < 2 < 3 < 4 < 5, que conserva la comparación (`>`, `sort_values`)
pero bloquea la aritmética sin sentido.

`sop` y `lor` son también escalas ordinales, pero con pasos de 0.5; se dejan como
`Float64` porque en la práctica se tratan como numéricas en el modelo y porque una
categórica con 9 niveles no aporta nada aquí. Es una decisión revisable en
`4-feat_eng`.

In [15]:
COLS_CATEGORICAS_ORDINALES = {"university_rating": ["1", "2", "3", "4", "5"]}

for columna, orden in COLS_CATEGORICAS_ORDINALES.items():
    df_tipado[columna] = pd.Categorical(df_tipado[columna], categories=orden, ordered=True)

print(df_tipado["university_rating"].dtype)
df_tipado["university_rating"].value_counts(dropna=False).sort_index()

category


university_rating
1       28
2      121
3      158
4       83
5       75
NaN      6
Name: count, dtype: int64

### 🔢 Variables numéricas

Dos grupos, según si el puntaje admite decimales:

- **Enteras** (`Int64`): `gre_score` y `toefl_score` son puntajes de examen que solo
  toman valores enteros.
- **Flotantes** (`Float64`): `sop`, `lor`, `cgpa` y `chance_of_admit` (la variable
  objetivo), con pasos de 0.5 o continuas.

In [16]:
COLS_ENTERAS = ["gre_score", "toefl_score"]
COLS_FLOTANTES = ["sop", "lor", "cgpa", "chance_of_admit"]

for columna in COLS_ENTERAS:
    df_tipado[columna] = pd.to_numeric(df_tipado[columna], errors="coerce").astype("Int64")

for columna in COLS_FLOTANTES:
    df_tipado[columna] = pd.to_numeric(df_tipado[columna], errors="coerce").astype("Float64")

df_tipado[COLS_ENTERAS + COLS_FLOTANTES].dtypes

gre_score            Int64
toefl_score          Int64
sop                Float64
lor                Float64
cgpa               Float64
chance_of_admit    Float64
dtype: object

### ✅ Variables booleanas

`research` es un indicador 0/1 de experiencia en investigación. Como `boolean` dice
explícitamente que solo hay dos estados posibles (más `pd.NA`), y así ningún modelo ni
resumen lo trata por error como una cantidad que se puede promediar como puntaje.

In [17]:
COLS_BOOLEANAS = ["research"]
MAPA_BOOLEANO = {"0": False, "1": True}

for columna in COLS_BOOLEANAS:
    valores_inesperados = set(df_tipado[columna].dropna().unique()) - set(MAPA_BOOLEANO)
    if valores_inesperados:
        raise ValueError(f"Valores no booleanos en '{columna}': {valores_inesperados}")
    df_tipado[columna] = df_tipado[columna].map(MAPA_BOOLEANO).astype("boolean")

print(df_tipado["research"].dtype)
df_tipado["research"].value_counts(dropna=False)

boolean


research
True     237
False    205
<NA>      29
Name: count, dtype: Int64

### 🔍 Verificación de la conversión

Si `errors="coerce"` hubiera descartado algún valor mal escrito, aparecería aquí como
un nulo nuevo. La comprobación no puede omitirse: es el único punto donde se detecta
que la conversión perdió información en silencio.

In [18]:
reporte = pd.DataFrame(
    {
        "dtype_final": df_tipado.dtypes.astype(str),
        "nulos_antes": nulos_antes,
        "nulos_despues": df_tipado.isna().sum(),
    }
)
reporte["perdidos_en_conversion"] = reporte["nulos_despues"] - reporte["nulos_antes"]
reporte

,dtype_final,nulos_antes,nulos_despues,perdidos_en_conversion
gre_score,Int64,11,11,0
toefl_score,Int64,19,19,0
university_rating,category,6,6,0
sop,Float64,16,16,0
lor,Float64,8,8,0
cgpa,Float64,2,2,0
research,boolean,29,29,0
chance_of_admit,Float64,0,0,0


In [19]:
perdidos = reporte.loc[reporte["perdidos_en_conversion"] > 0].index.tolist()
if perdidos:
    for columna in perdidos:
        malos = df.loc[df[columna].notna() & df_tipado[columna].isna(), columna]
        print(f"--- {columna}: {len(malos)} valores no convertidos ---")
        print(malos.value_counts().head(10).to_string())
else:
    print("Ninguna conversion descarto valores.")

Ninguna conversion descarto valores.


## ✅ Verificación del resultado

In [20]:
df_tipado.info()

<class 'pandas.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   gre_score          460 non-null    Int64   
 1   toefl_score        452 non-null    Int64   
 2   university_rating  465 non-null    category
 3   sop                455 non-null    Float64 
 4   lor                463 non-null    Float64 
 5   cgpa               469 non-null    Float64 
 6   research           442 non-null    boolean 
 7   chance_of_admit    471 non-null    Float64 
dtypes: Float64(4), Int64(2), boolean(1), category(1)
memory usage: 26.6 KB


In [21]:
# validacion de dominio: cada columna dentro del rango documentado en Informacion.txt
RANGOS = {
    "gre_score": (0, 340),
    "toefl_score": (0, 120),
    "sop": (1, 5),
    "lor": (1, 5),
    "cgpa": (0, 10),
    "chance_of_admit": (0, 1),
}

fuera_de_rango = {}
for columna, (minimo, maximo) in RANGOS.items():
    serie = df_tipado[columna].dropna()
    invalidos = serie[(serie < minimo) | (serie > maximo)]
    if len(invalidos):
        fuera_de_rango[columna] = invalidos.tolist()[:5]

if fuera_de_rango:
    raise ValueError(f"Valores fuera del dominio documentado: {fuera_de_rango}")

categorias_declaradas = set(COLS_CATEGORICAS_ORDINALES["university_rating"])
if not set(df_tipado["university_rating"].dropna().astype(str)) <= categorias_declaradas:
    raise ValueError("Aparecieron categorias fuera del conjunto declarado")

print("Todos los valores estan dentro del dominio documentado.")

Todos los valores estan dentro del dominio documentado.


In [22]:
# controles de calidad: si algo fallo, el notebook debe romperse aqui
if len(df_tipado) != len(df):
    raise ValueError("Se perdieron filas durante la conversion")
if not df_tipado.columns.is_unique:
    raise ValueError("Hay nombres de columna duplicados")

sin_tipar = [columna for columna in df_tipado.columns if df_tipado[columna].dtype == "object"]
if sin_tipar:
    raise TypeError(f"Columnas sin tipo definido: {sin_tipar}")
if df_tipado["chance_of_admit"].isna().any():
    raise ValueError("La variable objetivo no puede tener nulos")

# la comparacion de memoria incluye el efecto de eliminar los duplicados
memoria_antes = df_inferido.memory_usage(deep=True).sum() / 1024
memoria_despues = df_tipado.memory_usage(deep=True).sum() / 1024
print(
    f"Memoria: {memoria_antes:.1f} KB ({len(df_inferido)} filas) -> {memoria_despues:.1f} KB ({len(df_tipado)} filas)"
)
print("Controles de calidad superados.")

Memoria: 39.1 KB (623 filas) -> 26.6 KB (471 filas)
Controles de calidad superados.


In [23]:
df_tipado.describe(include="all").transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
gre_score,460.0,<NA>,<NA>,<NA>,316.458696,11.612687,290.0,308.0,316.5,325.0,340.0
toefl_score,452.0,<NA>,<NA>,<NA>,107.464602,6.129141,92.0,103.0,107.0,112.0,120.0
university_rating,465,5,3,158,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sop,455.0,<NA>,<NA>,<NA>,3.406593,1.010714,1.0,2.5,3.5,4.0,5.0
lor,463.0,<NA>,<NA>,<NA>,3.466523,0.897605,1.0,3.0,3.5,4.0,5.0
cgpa,469.0,<NA>,<NA>,<NA>,8.593475,0.594859,6.8,8.17,8.6,9.06,9.92
research,442,2,True,237,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chance_of_admit,471.0,<NA>,<NA>,<NA>,0.723588,0.143712,0.34,0.64,0.73,0.835,0.97


In [24]:
df_tipado.head()

,gre_score,toefl_score,university_rating,sop,lor,cgpa,research,chance_of_admit
0,337,118,4,4.5,4.5,9.65,True,0.92
1,324,107,4,4.0,4.5,8.87,True,0.76
2,316,104,3,3.0,3.5,8.0,True,0.72
3,322,110,3,3.5,2.5,8.67,True,0.8
4,314,103,2,2.0,3.0,8.21,False,0.65


## 💾 Almacenar el dataset en formato `.parquet`

El resultado va a `data/02_intermediate/`, que es exactamente la capa que la
convención de ingeniería de datos del proyecto define como *"los modelos de datos que
se introducen para tipar los datos crudos"*.

`.parquet` es el formato adecuado aquí porque es columnar, comprimido y —a diferencia
de CSV— **conserva el esquema**: al releerlo no hay que repetir ninguna de las
conversiones hechas en este notebook. Un CSV devolvería todo a texto y `research`
volvería a ser un `0/1` ambiguo.

In [25]:
esquema = pa.Table.from_pandas(df_tipado, preserve_index=False).schema

df_tipado.to_parquet(
    ARCHIVO_SALIDA,
    index=False,
    engine="pyarrow",
    compression="snappy",
    schema=esquema,
)

print(f"Guardado en: {ARCHIVO_SALIDA.relative_to(ARCHIVO_SALIDA.parents[2])}")
print(f"Tamano     : {ARCHIVO_SALIDA.stat().st_size / 1024:.1f} KB")
print(esquema)

Guardado en: data/02_intermediate/admisiones_type_fixed.parquet
Tamano     : 8.7 KB
gre_score: int64
toefl_score: int64
university_rating: dictionary<values=large_string, indices=int8, ordered=1>
sop: double
lor: double
cgpa: double
research: bool
chance_of_admit: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1065


In [26]:
# verificar que el archivo se relee con la misma forma y los mismos tipos
df_verificacion = pd.read_parquet(ARCHIVO_SALIDA)

if df_verificacion.shape != df_tipado.shape:
    raise ValueError("Cambio la forma del dataset al releer el parquet")

diferencias = {
    columna: (str(df_tipado[columna].dtype), str(df_verificacion[columna].dtype))
    for columna in df_tipado.columns
    if str(df_tipado[columna].dtype) != str(df_verificacion[columna].dtype)
}
if diferencias:
    raise TypeError(f"Tipos que no se conservaron en el parquet: {diferencias}")

print("El parquet conserva forma y tipos.")
df_verificacion.dtypes

El parquet conserva forma y tipos.


gre_score               Int64
toefl_score             Int64
university_rating    category
sop                   Float64
lor                   Float64
cgpa                  Float64
research              boolean
chance_of_admit       Float64
dtype: object

## 📊 Analysis of Results and Conclusions

**Estado inicial:** 623 filas x 8 columnas, sin columna identificadora, con todo
leído como texto (o como `float64` si se deja inferir a pandas).

**Problemas encontrados y qué se hizo con cada uno:**

| Problema | Evidencia | Acción |
|---|---|---|
| Nombres de columna con espacio final | `'LOR '`, `'Chance of Admit '` | Normalizados a `snake_case` |
| Nulos escritos como campo vacío | 91 celdas en 7 columnas, 71 filas afectadas | Unificados en `pd.NA` |
| Duplicados exactos | 152 filas (24 %), ninguna con nulos | Eliminados, quedan 471 filas |
| Tipos genéricos, no uniformes | Todo `object` o `float64` | `Int64`, `Float64`, `category` ordinal, `boolean` |
| Indicador 0/1 tratado como número | `research` | Convertido a `boolean` |
| Escala ordinal 1–5 tratada como número | `university_rating` | Convertido a categórica **ordinal** |

**Resultado:** ninguna conversión descartó valores (0 pérdidas silenciosas), todos los
valores caen dentro del dominio documentado en `Informacion.txt` y el dataset queda
guardado en `data/02_intermediate/admisiones_type_fixed.parquet` con su esquema.

**Lo que hay que tener presente en el siguiente paso:**

- La eliminación de duplicados es la decisión más fuerte del notebook y la única que
  no se puede *demostrar* con los datos disponibles (no hay identificador). Está
  aislada en la constante `ELIMINAR_DUPLICADOS` para poder revertirla.
- Quedan nulos sin imputar en 7 columnas, a propósito: la estrategia se decide en el
  EDA, no antes de ver las distribuciones.
- Los valores están dentro de rango, lo que descarta errores de captura groseros, pero
  **no** descarta atípicos legítimos: eso se mira en el análisis univariable.

## 💡 Proposals and Ideas

1. **Análisis exploratorio (`3-analysis`):** distribuciones univariables, relación de
   cada variable con `chance_of_admit` y correlaciones. Es de esperar que `cgpa`,
   `gre_score` y `toefl_score` estén fuertemente correlacionadas entre sí
   (multicolinealidad), lo que condiciona la elección del modelo.
2. **Estrategia de nulos:** comprobar si los faltantes son aleatorios (MCAR) o si se
   concentran en algún perfil, antes de elegir entre imputar por la mediana, usar un
   modelo que admita nulos o descartar filas.
3. **Reproducibilidad:** cuando estos pasos se estabilicen, moverlos de este notebook a
   una función en `src/` para que la limpieza sea parte del pipeline y no de una
   ejecución manual.
4. **Enfoque del problema:** `chance_of_admit` es continua, así que el problema es de
   regresión; pero si la decisión de negocio es *"segura / probable / ambiciosa"*, vale
   la pena evaluar también una discretización del objetivo en el notebook de modelos.

## 📖 References

- Estructura y pasos de exploración: <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/>
- Convención de capas de datos (Kedro): <https://docs.kedro.org/en/stable/faq/faq.html>
- Tipos de datos *nullable* en pandas: <https://pandas.pydata.org/docs/user_guide/integer_na.html>
- Datos categóricos ordinales en pandas: <https://pandas.pydata.org/docs/user_guide/categorical.html>
- Descripción del dataset: `data/01_raw/Informacion.txt`